# SustAInable — Notebook 03: Model Training & Evaluation
**Project:** SustAInable — Neighborhood Heat Illness Risk Prediction by ZIP Code  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/sustainable-heat](https://github.com/meyeringn/sustainable-heat)

---

## What This Notebook Does

1. Trains an **XGBoost** binary classifier on the SMOTE-balanced training data
2. Evaluates on the locked holdout set using **F2-Score**, AUC-ROC, and Precision-Recall curves
3. Produces the **ZIP-level risk score list** — the operational output public health agencies would use
4. Generates the confusion matrix showing the cost of each error type

### Why Recall Matters More Here

A missed high-risk ZIP code means a community doesn't receive targeted cooling center outreach, hydration distribution, or wellness checks during a heat event. In a severe heat wave, that can mean preventable deaths. A false alarm means outreach goes to a lower-risk neighborhood — wasteful, but not catastrophic.

**F2-Score weights recall twice as heavily as precision.** That is the right trade-off when the cost of missing a vulnerable community is measured in lives.

In [ ]:
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, fbeta_score, average_precision_score
)

np.random.seed(42)
sns.set_theme(style='whitegrid')
THRESHOLD = 0.30
print('Libraries loaded.')

## 1. Load Data & Train Model

In [ ]:
X_train = np.load('sustainable_X_train.npy')
y_train = np.load('sustainable_y_train.npy')
X_holdout = np.load('sustainable_X_holdout.npy')
y_holdout = np.load('sustainable_y_holdout.npy')
feature_cols = joblib.load('sustainable_feature_cols.pkl')

model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, use_label_encoder=False
)
model.fit(X_train, y_train, eval_set=[(X_holdout, y_holdout)], verbose=50)
joblib.dump(model, 'sustainable_xgboost_model.pkl')
print('\nModel saved: sustainable_xgboost_model.pkl')

## 2. Evaluate at Threshold = 0.30

In [ ]:
y_prob = model.predict_proba(X_holdout)[:, 1]
y_pred = (y_prob >= THRESHOLD).astype(int)

auc_roc = roc_auc_score(y_holdout, y_prob)
f2 = fbeta_score(y_holdout, y_pred, beta=2)
avg_prec = average_precision_score(y_holdout, y_prob)

print(f'=== SustAInable Model — Holdout Evaluation ===')
print(f'Decision Threshold: {THRESHOLD}')
print(f'AUC-ROC:            {auc_roc:.4f}')
print(f'F2-Score:           {f2:.4f}  ← Primary metric')
print(f'Avg Precision:      {avg_prec:.4f}')
print()
print(classification_report(y_holdout, y_pred, target_names=['Lower Risk', 'Elevated Risk']))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_holdout, y_pred)
annot = np.array([
    [f'True Negative\n{cm[0,0]}\n(Correctly lower risk)', f'False Positive\n{cm[0,1]}\n(Unnecessary outreach)'],
    [f'FALSE NEGATIVE\n{cm[1,0]}\n⚠ Vulnerable community missed', f'True Positive\n{cm[1,1]}\n(Intervention delivered)']
])
sns.heatmap(cm, annot=annot, fmt='', cmap='Reds', ax=ax,
            xticklabels=['Predicted: Lower Risk', 'Predicted: Elevated Risk'],
            yticklabels=['Actual: Lower Risk', 'Actual: Elevated Risk'],
            linewidths=2, linecolor='white')
ax.set_title(f'SustAInable Confusion Matrix\nThreshold = {THRESHOLD} | F2-Score = {f2:.3f}',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('sustainable_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_confusion_matrix.png')

In [ ]:
# PR + ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('SustAInable — Model Performance Curves', fontweight='bold', fontsize=14)

prec, rec, pr_thresh = precision_recall_curve(y_holdout, y_prob)
axes[0].plot(rec, prec, color='#C62828', lw=2.5, label=f'AP = {avg_prec:.3f}')
axes[0].axhline(y=y_holdout.mean(), color='gray', linestyle='--', alpha=0.7, label='Baseline')
idx = np.argmin(np.abs(pr_thresh - THRESHOLD))
axes[0].scatter(rec[idx], prec[idx], s=180, color='#1565C0', zorder=5, label=f'Threshold={THRESHOLD}')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve', fontweight='bold')
axes[0].legend(); axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])

fpr, tpr, roc_thresh = roc_curve(y_holdout, y_prob)
axes[1].plot(fpr, tpr, color='#2E7D32', lw=2.5, label=f'AUC = {auc_roc:.3f}')
axes[1].plot([0,1],[0,1],'k--', alpha=0.5, label='Random')
idx_r = np.argmin(np.abs(roc_thresh - THRESHOLD))
axes[1].scatter(fpr[idx_r], tpr[idx_r], s=180, color='#1565C0', zorder=5, label=f'Threshold={THRESHOLD}')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(); axes[1].set_xlim([0,1]); axes[1].set_ylim([0,1])

plt.tight_layout()
plt.savefig('sustainable_performance_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_performance_curves.png')

## 3. ZIP-Level Risk Score Output

In [ ]:
df_orig = pd.read_csv('sustainable_zip_data.csv')
encoders = joblib.load('sustainable_encoders.pkl')
df_score = df_orig.copy()
df_score['region_encoded'] = encoders['region'].transform(df_score['region'])
df_score['urban_type_encoded'] = encoders['urban_type'].transform(df_score['urban_type'])

# Recreate engineered features
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
vuln = df_score[['poverty_rate_pct','pct_65_plus','uninsured_rate_pct',
                  'prev_heart_disease_pct','prev_diabetes_pct']].values
df_score['social_vulnerability_index'] = scaler.fit_transform(vuln).mean(axis=1)
df_score['infrastructure_gap'] = (
    df_score['extreme_heat_days_annual'] / df_score['extreme_heat_days_annual'].max() -
    df_score['ac_access_pct'] / 100 -
    df_score['cooling_centers_per_10k'] / df_score['cooling_centers_per_10k'].max()
)
df_score['heat_x_vulnerability'] = df_score['avg_summer_temp_f'] / 100 * df_score['social_vulnerability_index']
df_score['canopy_deficit'] = df_score['impervious_surface_pct'] - df_score['tree_canopy_coverage_pct']

full_probs = model.predict_proba(df_score[feature_cols].values)[:, 1]

risk_df = df_orig[['zip_code','region','urban_type','avg_summer_temp_f',
                    'extreme_heat_days_annual','poverty_rate_pct','ac_access_pct']].copy()
risk_df['risk_score'] = full_probs.round(4)
risk_df['risk_flag'] = (full_probs >= THRESHOLD).astype(int)
risk_df['priority_tier'] = pd.cut(
    full_probs, bins=[0,0.30,0.60,0.80,1.0],
    labels=['Monitor','Elevated','High','Critical'])
risk_df = risk_df.sort_values('risk_score', ascending=False).reset_index(drop=True)
risk_df.to_csv('sustainable_risk_scores.csv', index=False)

print(f'Saved: sustainable_risk_scores.csv')
print(f'ZIP codes flagged: {risk_df["risk_flag"].sum()}')
print(f'\nTier breakdown:')
print(risk_df['priority_tier'].value_counts())
print(f'\nTop 10 highest-risk ZIP codes:')
risk_df.head(10)

## Summary

| Output | Description |
|---|---|
| `sustainable_xgboost_model.pkl` | Trained model |
| `sustainable_confusion_matrix.png` | Error type visualization |
| `sustainable_performance_curves.png` | PR and ROC curves |
| `sustainable_risk_scores.csv` | Ranked ZIP code risk list |

➡️ **Next:** Notebook 04 — SHAP Explainability